In [0]:
df_master = spark.read.table("customer_master")
df_master.show(5)
print("shape:",df_master.count(),len(df_master.columns))

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [0]:
from pyspark.sql.functions import col,sum as _sum

null_counts = df_master.select([_sum(col(c).isNull().cast("int")).alias(c) for c in df_master.columns])
null_counts.show()

df_clean = df_master.dropDuplicates().dropna(subset=["Order id","Customer ID"])

print("Before cleaning:",df_master.count())
print("After cleaning:",df_clean.count())

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+

Before cl

In [0]:
from pyspark.sql.functions import col

new_cols = [c.replace(' ', '_') for c in df_clean.columns]
df_clean_fixed = df_clean.toDF(*new_cols)

df_clean_fixed.write.format("delta").mode("overwrite").saveAsTable("customer_master_delta")
print("Delta table created successfully")
print("Total rows in Delta table:", spark.read.table("customer_master_delta").count())

Delta table created successfully
Total rows in Delta table: 15227


In [0]:
from pyspark.sql.functions import lit

df_updates = df_clean_fixed.limit(100)

new_data = [
    ("9999", "NEW-001", "7/29/2026", "8/1/2026", "First Class", "NEW-001", "New Customer 1", "Consumer", "United States", "New York", "New York", "10001", "East", "OFF-001", "Office Supplies", "Pens", "Premium Pen Set", 50.0, 2, 0.0, 10.0),
    ("9998", "NEW-002", "7/29/2026", "8/1/2026", "Second Class", "NEW-002", "New Customer 2", "Corporate", "United States", "Chicago", "Illinois", "60601", "Central", "OFF-002", "Office Supplies", "Paper", "Premium Paper", 30.0, 5, 0.0, 8.0),
]

df_new = spark.createDataFrame(new_data, df_clean_fixed.columns)

df_incremental = df_updates.union(df_new)
print("Incremental records:", df_incremental.count())
df_incremental.show(5)

Incremental records: 102
+------+--------------+----------+----------+--------------+-----------+---------------+--------+-------------+-----------+----------+-----------+-------+---------------+---------------+------------+--------------------+-------+--------+--------+--------+
|Row_ID|      Order_ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|  Customer_Name| Segment|      Country|       City|     State|Postal_Code| Region|     Product_ID|       Category|Sub-Category|        Product_Name|  Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+--------+-------------+-----------+----------+-----------+-------+---------------+---------------+------------+--------------------+-------+--------+--------+--------+
|     9|CA-2014-115812|  6/9/2014| 6/14/2014|Standard Class|   BH-11710|Brosina Hoffman|Consumer|United States|Los Angeles|California|      90032|   West|OFF-BI-10003910|Office Supplies|     Binders|DXL Angl

In [0]:
from delta.tables import DeltaTable

# Remove duplicate Order_IDs from incremental data before MERGE
df_incremental_dedup = df_incremental.dropDuplicates(["Order_ID"])
print("Deduplicated incremental records:", df_incremental_dedup.count())

# Load the Delta table
delta_table = DeltaTable.forName(spark, "customer_master_delta")

# Apply MERGE operation
delta_table.alias("master").merge(
    df_incremental_dedup.alias("incremental"),
    "master.Order_ID = incremental.Order_ID"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("MERGE operation completed successfully")

Deduplicated incremental records: 96
MERGE operation completed successfully


In [0]:
df_final = spark.read.table("customer_master_delta")

print(" VALIDATION RESULTS ")
print("Total rows after MERGE:", df_final.count())
print("Duplicate Order_IDs:", df_final.groupBy("Order_ID").count().filter("count > 1").count())
df_final.show(10)

 VALIDATION RESULTS 
Total rows after MERGE: 15229
Duplicate Order_IDs: 3819
+------+--------------+----------+----------+--------------+-----------+--------------------+---------+-------------+-----------+----------+-----------+-------+---------------+---------------+------------+--------------------+-------+--------+--------+--------+
|Row_ID|      Order_ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|       Customer_Name|  Segment|      Country|       City|     State|Postal_Code| Region|     Product_ID|       Category|Sub-Category|        Product_Name|  Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+--------------------+---------+-------------+-----------+----------+-----------+-------+---------------+---------------+------------+--------------------+-------+--------+--------+--------+
|    51|CA-2015-115742| 4/18/2015| 4/22/2015|Standard Class|   DP-13000|       Darren Powers| Consumer|United States| New Albany|   India

In [0]:
print("=" * 50)
print("   DELTA LAKE MERGE ASSIGNMENT - SUMMARY")
print("=" * 50)
print(f"Original records loaded:       19,988")
print(f"After cleaning (duplicates):   15,227")
print(f"Incremental records processed: 96")
print(f"New records inserted:          2")
print(f"Final records after MERGE:     {df_final.count()}")
print(f"Columns in dataset:            {len(df_final.columns)}")
print("=" * 50)
print("MERGE Operation: SUCCESS")
print("Delta Lake Implementation: COMPLETE ")

   DELTA LAKE MERGE ASSIGNMENT - SUMMARY
Original records loaded:       19,988
After cleaning (duplicates):   15,227
Incremental records processed: 96
New records inserted:          2
Final records after MERGE:     15229
Columns in dataset:            21
MERGE Operation: SUCCESS
Delta Lake Implementation: COMPLETE 
